# HomeWork #4 - MapReduce

Implementazione con MapReduce

In [42]:
import csv
import random
from datetime import datetime, timedelta
from pathlib import Path

NUM_EXAMPLES = 200_000
FOLDER_DATASET = "dataset"
FILENAME_DATASET = "pedaggio.csv"
HEADER = [
    "IDVeicolo",
    "TipoVeicolo",
    "Tratta",
    "Pedaggio",
    "DataTransito",
    "FasciaOraria",
    "Provincia",
]

TIPO_VEICOLO = ["AutoTermiche", "AutoElettriche", "Moto", "Camion", "Bus", "Furgone"]

# Mappatura tratta -> province attraversate.
TRATTA_PROVINCE = {
    "A1": ["Milano", "Lodi", "Piacenza", "Parma", "Modena", "Bologna", "Firenze", "Arezzo", "Roma", "Napoli"],
    "A4": ["Torino", "Novara", "Milano", "Bergamo", "Brescia", "Verona", "Vicenza", "Padova", "Venezia", "Trieste"],
    "A7": ["Milano", "Pavia", "Genova"],
    "A10": ["Genova", "Savona", "Imperia"],
    "A12": ["Genova", "La Spezia", "Massa-Carrara", "Livorno", "Roma"],
    "A14": ["Bologna", "Forli-Cesena", "Rimini", "Pesaro-Urbino", "Ancona", "Pescara", "Chieti", "Foggia", "Bari", "Taranto"],
    "A21": ["Torino", "Asti", "Alessandria", "Piacenza", "Brescia"],
    "A22": ["Modena", "Mantova", "Verona", "Trento", "Bolzano"],
    "A23": ["Udine", "Gorizia", "Trieste"],
    "A24": ["Roma", "L'Aquila", "Teramo"],
    "A26": ["Genova", "Alessandria", "Vercelli", "Verbano-Cusio-Ossola"],
    "A30": ["Caserta", "Salerno"],
    "A32": ["Torino"],
    "A90": ["Roma"],
    "A91": ["Roma"],
    "A19": ["Palermo", "Caltanissetta", "Enna", "Catania"],
    "A18": ["Messina", "Catania", "Siracusa"],
    "A20": ["Messina", "Palermo"],
}

TRATTE = list(TRATTA_PROVINCE.keys())

BASE_PEDAGGIO = {
    "AutoTermiche": (8, 22),
    "AutoElettriche": (5, 16),
    "Moto": (4, 14),
    "Camion": (18, 55),
    "Bus": (20, 60),
    "Furgone": (12, 35),
}

ANNO_BASE = 2015
ANNO_FINE = 2025
TASSO_INFLAZIONE_ANNUO = 0.10

def get_fattore_inflazione(anno):
    return (1 + TASSO_INFLAZIONE_ANNUO) ** (anno - ANNO_BASE)

def get_pedaggio(tipo_veicolo, tratta, anno):
    minimo, massimo = BASE_PEDAGGIO[tipo_veicolo]
    fattore_tratta = max(1, len(TRATTA_PROVINCE[tratta]) // 3)
    pedaggio_base = random.randint(minimo, massimo) + fattore_tratta
    pedaggio = pedaggio_base * get_fattore_inflazione(anno)
    return round(pedaggio, 2)

def get_data_transito():
    start = datetime(ANNO_BASE, 1, 1)
    end = datetime(ANNO_FINE, 12, 31, 23, 59, 59)
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    data_transito = start + timedelta(seconds=random_seconds)
    return data_transito, data_transito.strftime("%Y-%m-%d %H:%M:%S")

def get_fascia_oraria(data_transito):
    ora = data_transito.hour
    if ora < 6:
        return "00:00-06:00"
    if ora < 12:
        return "06:00-12:00"
    if ora < 18:
        return "12:00-18:00"
    return "18:00-24:00"

def genera_record(index):
    tipo_veicolo = random.choice(TIPO_VEICOLO)
    tratta = random.choice(TRATTE)
    provincia = random.choice(TRATTA_PROVINCE[tratta])
    data_transito, data_transito_str = get_data_transito()
    return {
        "IDVeicolo": f"V{index:06d}",
        "TipoVeicolo": tipo_veicolo,
        "Tratta": tratta,
        "Pedaggio": get_pedaggio(tipo_veicolo, tratta, data_transito.year),
        "DataTransito": data_transito_str,
        "FasciaOraria": get_fascia_oraria(data_transito),
        "Provincia": provincia,
    }

dataset_path = Path(FOLDER_DATASET) / FILENAME_DATASET

print(f"Vuoi rigenerare il dataset? (Attualmente: {FILENAME_DATASET}) [y/N]")
if input().strip().lower() == 'y':
    with dataset_path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=HEADER, delimiter=";")
        writer.writeheader()
        for index in range(1, NUM_EXAMPLES + 1):
            writer.writerow(genera_record(index))
    print(f"Dataset generated: {NUM_EXAMPLES} righe")

Vuoi rigenerare il dataset? (Attualmente: pedaggio.csv) [y/N]


## Consegna
Il programma MapReduce deve calcolare il pedaggio medio pagato da ogni tipo di veicolo (es.
camion) nell’anno 2025 e come questo è cambiato rispetto a 10 anni fa. Il campo Data è nel
formato GG/MM/YYYY.

### File MR con Combiner

In [43]:
%%writefile main_mrjob_combiner.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MediaPedaggi(MRJob):

    def mapper(self, _, line):
        # Implementa la logica del mapper
        if line.startswith("IDVeicolo"):
            return  # Salta l'intestazione
        
        fields = line.split(";")
        if len(fields) != 7:
            return  # Salta righe malformate
        
        try:
            tipoVeicolo = fields[1]
            pedaggio = float(fields[3])
            anno = int(fields[4].split("-")[0])  # Estrai l'anno dalla data

            if anno == 2015 or anno == 2025:
                yield tipoVeicolo, (anno, pedaggio)
        except (ValueError, IndexError):
            return  # Salta righe con dati non validi

        

    def combiner(self, tipoVeicolo, values):
        # Logica combiner
        sum_2015=0.0
        count_2015=0
        sum_2025=0.0
        count_2025=0

        for anno, pedaggio in values:
            if anno == 2015:
                count_2015 += 1
                sum_2015 += pedaggio
            elif anno == 2025:
                count_2025 += 1
                sum_2025 += pedaggio
        yield tipoVeicolo, ("combiner", sum_2015, count_2015, sum_2025, count_2025)

    def reducer(self, tipoVeicolo, values):
        # Logica reducer
        sum_2015=0
        count_2015=0
        sum_2025=0
        count_2025=0

        for value in values:
            if value[0] == "combiner": # Logica con combiner
                sum_2015 += value[1]
                count_2015 += value[2]
                sum_2025 += value[3]
                count_2025 += value[4]
            else:
                anno, pedaggio = value # Logica senza combiner
                if anno == 2015:
                    count_2015 += 1
                    sum_2015 += pedaggio
                elif anno == 2025:
                    count_2025 += 1
                    sum_2025 += pedaggio
        
        average_2015 = sum_2015 / count_2015 if count_2015 > 0 else 0
        average_2025 = sum_2025 / count_2025 if count_2025 > 0 else 0
        variazione = average_2025 - average_2015
        variazione_percentuale = (variazione / average_2015 * 100) if average_2015 > 0 else 0
        
        yield tipoVeicolo, {
            "average_2015": average_2015,
            "average_2025": average_2025,
            "variazione": round(variazione, 2),
            "variazione_percentuale": round(variazione_percentuale, 2),
        }

if __name__ == '__main__':
    MediaPedaggi.run()

Overwriting main_mrjob_combiner.py


### Testing con Combiner

In [47]:
import os
import sys
import time

# Aggiungo la cartella corrente al path, così Python trova il modulo
sys.path.append(os.getcwd())


def stampa_separatore(w_tipo, w_2015, w_2025, w_var, w_varp):
    print(
        f"+{'-'*(w_tipo+2)}+{'-'*(w_2015+2)}+{'-'*(w_2025+2)}+{'-'*(w_var+2)}+{'-'*(w_varp+2)}+"
    )


def stampa_tabella_risultati(job, runner):
    found_data = False

    header = f"{'TipoVeicolo':<18} {'Media 2015':>12} {'Media 2025':>12} {'Var. Ass.':>12} {'Var. %':>12}"
    print("\nRisultati finali")
    print("-" * len(header))
    print(header)
    print("-" * len(header))

    for key, value in job.parse_output(runner.cat_output()):
        found_data = True

        print(
            f"{key:<18} "
            f"{value['average_2015']:>12.2f} "
            f"{value['average_2025']:>12.2f} "
            f"{value['variazione']:>12.2f} "
            f"{value['variazione_percentuale']:>11.2f}%"
        )

    print("-" * len(header))

    if not found_data:
        print("Nessun dato in output.")




try:
    from main_mrjob_combiner import MediaPedaggi

    file_input = "dataset/pedaggio.csv"

    print("--- AVVIO ANALISI MAPREDUCE ---\n")
    print("Modulo caricato correttamente.")
    print("Esecuzione in corso...")

    job = MediaPedaggi(args=[file_input])

    with job.make_runner() as runner:
        start_time = time.time()
        runner.run()
        elapsed_time = time.time() - start_time

        print("\nJob completato")
        print(f"Tempo effettivo di calcolo mrjob: {elapsed_time:.4f} secondi")
        print("=" * 70)

        stampa_tabella_risultati(job, runner)

except ImportError:
    print("Errore: non trovo il file 'main_mrjob.py' oppure la classe 'MediaPedaggi'.")

except FileNotFoundError:
    print("Errore: il file 'dataset/pedaggio.csv' non esiste.")

except Exception as e:
    print(f"Errore imprevisto: {e}")


No configs specified for inline runner


--- AVVIO ANALISI MAPREDUCE ---

Modulo caricato correttamente.
Esecuzione in corso...

Job completato
Tempo effettivo di calcolo mrjob: 1.5754 secondi

Risultati finali
----------------------------------------------------------------------
TipoVeicolo          Media 2015   Media 2025    Var. Ass.       Var. %
----------------------------------------------------------------------
AutoElettriche            11.85        30.67        18.82      158.80%
AutoTermiche              16.17        42.25        26.08      161.25%
Bus                       41.38       107.91        66.53      160.77%
Camion                    37.75        98.58        60.83      161.14%
Furgone                   24.79        64.08        39.29      158.46%
Moto                      10.30        26.90        16.60      161.16%
----------------------------------------------------------------------


### File senza combiner

In [45]:
%%writefile main_mrjob.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MediaPedaggi(MRJob):

    def mapper(self, _, line):
        # Implementa la logica del mapper
        if line.startswith("IDVeicolo"):
            return  # Salta l'intestazione
        
        fields = line.split(";")
        if len(fields) != 7:
            return  # Salta righe malformate
        
        try:
            tipoVeicolo = fields[1]
            pedaggio = float(fields[3])
            anno = int(fields[4].split("-")[0])  # Estrai l'anno dalla data

            if anno == 2015 or anno == 2025:
                yield tipoVeicolo, (anno, pedaggio)
        except (ValueError, IndexError):
            return  # Salta righe con dati non validi

    def reducer(self, tipoVeicolo, values):
        # Logica reducer
        sum_2015=0
        count_2015=0
        sum_2025=0
        count_2025=0

        for value in values:
            if value[0] == "combiner": # Logica con combiner
                sum_2015 += value[1]
                count_2015 += value[2]
                sum_2025 += value[3]
                count_2025 += value[4]
            else:
                anno, pedaggio = value # Logica senza combiner
                if anno == 2015:
                    count_2015 += 1
                    sum_2015 += pedaggio
                elif anno == 2025:
                    count_2025 += 1
                    sum_2025 += pedaggio
        
        average_2015 = sum_2015 / count_2015 if count_2015 > 0 else 0
        average_2025 = sum_2025 / count_2025 if count_2025 > 0 else 0
        variazione = average_2025 - average_2015
        variazione_percentuale = (variazione / average_2015 * 100) if average_2015 > 0 else 0
        
        yield tipoVeicolo, {
            "average_2015": average_2015,
            "average_2025": average_2025,
            "variazione": round(variazione, 2),
            "variazione_percentuale": round(variazione_percentuale, 2),
        }

if __name__ == '__main__':
    MediaPedaggi.run()

Overwriting main_mrjob.py


### Testing senza combiner

In [48]:
import os
import sys
import time

# Aggiungo la cartella corrente al path, così Python trova il modulo
sys.path.append(os.getcwd())


def stampa_separatore(w_tipo, w_2015, w_2025, w_var, w_varp):
    print(
        f"+{'-'*(w_tipo+2)}+{'-'*(w_2015+2)}+{'-'*(w_2025+2)}+{'-'*(w_var+2)}+{'-'*(w_varp+2)}+"
    )


def stampa_tabella_risultati(job, runner):
    found_data = False

    header = f"{'TipoVeicolo':<18} {'Media 2015':>12} {'Media 2025':>12} {'Var. Ass.':>12} {'Var. %':>12}"
    print("\nRisultati finali")
    print("-" * len(header))
    print(header)
    print("-" * len(header))

    for key, value in job.parse_output(runner.cat_output()):
        found_data = True

        print(
            f"{key:<18} "
            f"{value['average_2015']:>12.2f} "
            f"{value['average_2025']:>12.2f} "
            f"{value['variazione']:>12.2f} "
            f"{value['variazione_percentuale']:>11.2f}%"
        )

    print("-" * len(header))

    if not found_data:
        print("Nessun dato in output.")




try:
    from main_mrjob import MediaPedaggi

    file_input = "dataset/pedaggio.csv"

    print("--- AVVIO ANALISI MAPREDUCE ---\n")
    print("Modulo caricato correttamente.")
    print("Esecuzione in corso...")

    job = MediaPedaggi(args=[file_input])

    with job.make_runner() as runner:
        start_time = time.time()
        runner.run()
        elapsed_time = time.time() - start_time

        print("\nJob completato")
        print(f"Tempo effettivo di calcolo mrjob: {elapsed_time:.4f} secondi")
        print("=" * 70)

        stampa_tabella_risultati(job, runner)

except ImportError:
    print("Errore: non trovo il file 'main_mrjob.py' oppure la classe 'MediaPedaggi'.")

except FileNotFoundError:
    print("Errore: il file 'dataset/pedaggio.csv' non esiste.")

except Exception as e:
    print(f"Errore imprevisto: {e}")


No configs specified for inline runner


--- AVVIO ANALISI MAPREDUCE ---

Modulo caricato correttamente.
Esecuzione in corso...

Job completato
Tempo effettivo di calcolo mrjob: 1.4552 secondi

Risultati finali
----------------------------------------------------------------------
TipoVeicolo          Media 2015   Media 2025    Var. Ass.       Var. %
----------------------------------------------------------------------
AutoElettriche            11.85        30.67        18.82      158.80%
AutoTermiche              16.17        42.25        26.08      161.25%
Bus                       41.38       107.91        66.53      160.77%
Camion                    37.75        98.58        60.83      161.14%
Furgone                   24.79        64.08        39.29      158.46%
Moto                      10.30        26.90        16.60      161.16%
----------------------------------------------------------------------


## Considerazioni

### Considerazioni quantitative

Sono state confrontate due versioni dello stesso job MapReduce: una senza combiner e una con combiner. Dal punto di vista dei risultati finali, entrambe le soluzioni producono lo stesso output, cioè il pedaggio medio per tipo di veicolo nel 2015 e nel 2025, insieme alla variazione assoluta e percentuale. La differenza riguarda invece le prestazioni. Nei test eseguiti in locale con mrjob e runner inline, l’introduzione del combiner non ha portato necessariamente a un miglioramento dei tempi; in alcune prove la versione senza combiner è risultata persino leggermente più veloce. Questo comportamento è plausibile perché, in ambiente locale e su dataset di dimensione limitata, il costo aggiuntivo del combiner può superare il beneficio della riduzione dei dati intermedi.

### Considerazioni qualitative

La soluzione con combiner, invece, è più vicina alla logica MapReduce ottimizzata, perché introduce una pre-aggregazione locale dei dati emessi dal mapper, riducendo il numero di valori da inviare al reducer. In un contesto realmente distribuito, ad esempio su cluster Hadoop, questa strategia potrebbe ridurre il traffico di rete e migliorare la scalabilità. Nel caso analizzato, tuttavia, il vantaggio teorico del combiner non emerge in modo netto